# Notebook 4: Google Meridian - Setup & Data Formatting

## Overview

**Google Meridian** is Google's open-source Bayesian Marketing Mix Model (MMM) framework built on **JAX/NumPyro**. It provides a production-grade implementation of hierarchical Bayesian MMM with built-in support for:

- **Hill-Adstock** media transformations
- **Geo-level** hierarchical modeling
- **Budget optimization** via response curves
- **Prior calibration** from experiments

This notebook verifies your Meridian installation and formats the workshop data into the structure Meridian expects. We will use this formatted data in Session 5 for a full model walkthrough.

In [ ]:
# --- Installation Verification ---
try:
    import meridian
    from meridian.data.input_data import InputData
    from meridian.model import model as meridian_model
    from meridian.model import spec as meridian_spec
    print(f"Meridian version: {meridian.__version__}")
    print("Meridian imported successfully!")
except ImportError as e:
    print(f"Meridian not available: {e}")
    print("Install with: pip install google-meridian")

## Data Requirements

Meridian expects data in a very specific format. The core inputs are **matrices** with dimensions organized by **geo** and **time**:

| Input | Shape | Description |
|-------|-------|-------------|
| `media_spend` | `(n_geos, n_times, n_media_channels)` | Spend per channel per geo per time period |
| `media` (volume) | `(n_geos, n_media_times, n_media_channels)` | Impressions/GRPs (optional, defaults to spend) |
| `kpi` | `(n_geos, n_times)` | Target variable (revenue, conversions, etc.) |
| `population` | `(n_geos,)` | Population per geo (used for scaling) |
| `controls` | `(n_geos, n_times, n_controls)` | Control variables (seasonality, macro, etc.) |

**Single-geo case:** When you have national-level data (no geo breakdown), the geo dimension is simply 1. So a dataset with 36 months and 5 media channels would have `media_spend` shaped `(1, 36, 5)`.

**Time coordinates:** Meridian requires date-formatted strings (`YYYY-MM-DD`) as time coordinates.

Data is passed to Meridian via an **`InputData`** dataclass, which wraps xarray DataArrays. Each DataArray must have a `name` attribute matching its field name (e.g., `name='kpi'`, `name='media_spend'`).

In [ ]:
# --- Format Workshop Data ---
import numpy as np
import pandas as pd

# Load the workshop dataset
df = pd.read_excel('../../data/MMM_Workshop_Data.xlsx', sheet_name='Data')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# --- Identify columns ---
# Adjust these based on your actual column names
date_col = 'Month'  # time column in the workshop data
kpi_col = 'Sales_Revenue_Total'  # or 'Sales_Volume_Total', etc.

# Media spend columns (update to match your data)
media_cols = [col for col in df.columns if 'Spend' in col or 'spend' in col]
print(f"Detected media columns: {media_cols}")

# Control columns (everything that is not date, KPI, or media)
exclude_cols = [date_col, kpi_col] + media_cols
control_cols = [col for col in df.columns if col not in exclude_cols and df[col].dtype in ['float64', 'int64']]
print(f"Detected control columns: {control_cols}")

n_times = len(df)
n_geos = 1  # single-geo (national level)
n_media = len(media_cols)
n_controls = len(control_cols)

print(f"\nDimensions: {n_times} time periods, {n_geos} geo, {n_media} media channels, {n_controls} controls")

In [ ]:
# --- Create arrays in Meridian-expected shapes ---
# Meridian expects (n_geos, n_times, n_channels) for media
# and (n_geos, n_times) for KPI

# Media spend: (1, n_times, n_media)
media_spend_array = df[media_cols].values.reshape(n_geos, n_times, n_media)
print(f"media_spend shape: {media_spend_array.shape}")

# KPI: (1, n_times)
kpi_array = df[kpi_col].values.reshape(n_geos, n_times)
print(f"kpi shape: {kpi_array.shape}")

# Controls: (1, n_times, n_controls)
if n_controls > 0:
    controls_array = df[control_cols].values.reshape(n_geos, n_times, n_controls)
    print(f"controls shape: {controls_array.shape}")
else:
    controls_array = None
    print("No control variables detected.")

# Optional: media volume (impressions) - use spend as proxy if not available
media_volume_array = media_spend_array.copy()
print(f"\nAll arrays formatted successfully!")

In [ ]:
# --- Create Meridian InputData Object ---
import xarray as xr

try:
    # Build coordinate labels — Meridian requires YYYY-MM-DD date strings
    time_coords = [d.strftime('%Y-%m-%d') for d in pd.to_datetime(df[date_col])]
    geo_coords = ['national']
    media_channel_coords = media_cols

    # Create xarray DataArrays (each must have a name matching its InputData field)
    media_spend_da = xr.DataArray(
        media_spend_array, name='media_spend',
        dims=['geo', 'time', 'media_channel'],
        coords={'time': time_coords, 'geo': geo_coords, 'media_channel': media_channel_coords}
    )

    media_da = xr.DataArray(
        media_volume_array, name='media',
        dims=['geo', 'media_time', 'media_channel'],
        coords={'media_time': time_coords, 'geo': geo_coords, 'media_channel': media_channel_coords}
    )

    kpi_da = xr.DataArray(
        kpi_array, name='kpi',
        dims=['geo', 'time'],
        coords={'time': time_coords, 'geo': geo_coords}
    )

    population_da = xr.DataArray(
        [1.0], name='population',
        dims=['geo'],
        coords={'geo': geo_coords}
    )

    # Build InputData
    input_data = InputData(
        kpi=kpi_da,
        kpi_type='revenue',
        population=population_da,
        media_spend=media_spend_da,
        media=media_da,
    )

    print("InputData created successfully!")
    print(f"  Media channels: {list(input_data.media_spend.media_channel.values)}")
    print(f"  Time periods: {input_data.media_spend.sizes['time']}")
    print(f"  Geos: {input_data.media_spend.sizes['geo']}")

except Exception as e:
    print(f"Could not create InputData: {e}")
    print("The arrays above are correctly formatted - we will use them in Session 5.")

In [ ]:
# --- Minimal Model Spec ---
try:
    model_spec = meridian_spec.ModelSpec(
        max_lag=8,
    )

    print("ModelSpec created successfully!")
    print(f"\nModel Specification:")
    print(f"  Max lag: {model_spec.max_lag}")
    print(f"  Media prior type: {model_spec.media_prior_type}")
    print(f"  Adstock decay spec: {model_spec.adstock_decay_spec}")
    print(f"\nThis minimal spec uses Meridian's default priors for all parameters.")
    print("In Session 5, we will customize priors and run the full model.")

except Exception as e:
    print(f"Could not create ModelSpec: {e}")

## Troubleshooting

### Common Installation Issues

1. **JAX not found**: Meridian requires JAX. Install with:
   ```bash
   pip install jax jaxlib
   ```
   On Mac with Apple Silicon, you may need:
   ```bash
   pip install jax-metal
   ```

2. **NumPyro version conflict**: Meridian pins specific NumPyro versions. Use a dedicated virtual environment:
   ```bash
   python -m venv meridian_env
   source meridian_env/bin/activate
   pip install google-meridian
   ```

3. **Memory issues**: Meridian's MCMC sampling is memory-intensive. Ensure at least 8GB RAM available.

### Google Colab Fallback

If local installation fails, use Google Colab:

1. Open [Google Colab](https://colab.research.google.com)
2. Run: `!pip install google-meridian`
3. Upload the workshop data file
4. Copy the code cells from this notebook

Colab provides a pre-configured environment with JAX support and GPU access (which accelerates MCMC sampling significantly).

### Verifying Your Setup

If the import cell above ran without errors, you are ready for Session 5. If not, try the Colab fallback before the session.